# Embeddings and meta ensemble

Archived from the former Python scripts/package so the Hermes experiments are kept as notebooks.


## SVD and Embedding Chapter Model

Embedding-style and dimensionality-reduction experiments for chapter prediction.

_Archived from `src/icd10_coding/models/embedding_chapter.py`._


In [ ]:
"""Optional chapter model using multilingual sentence embeddings."""

import time

import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

from icd10_coding.paths import DATA_PROCESSED_DIR


MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def run() -> float:
    train = pd.read_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv")
    train["Literal_clean"] = train["Literal_clean"].fillna("")
    train["chapter"] = train["Code"].astype(str).str[0]

    train_split, val_split = train_test_split(train, test_size=0.2, random_state=42)

    print(f"Loading embedding model: {MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME)

    start = time.time()
    # Encode literals once, then train a small linear classifier on the dense vectors.
    x_train = model.encode(
        train_split["Literal_clean"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    x_val = model.encode(
        val_split["Literal_clean"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f"Encoded literals in {time.time() - start:.1f}s")

    clf = LinearSVC(C=2.0, class_weight="balanced", max_iter=5000, random_state=42)
    clf.fit(x_train, train_split["chapter"])
    pred = clf.predict(x_val)
    score = accuracy_score(val_split["chapter"], pred)
    print(f"Embedding chapter validation accuracy: {score * 100:.2f}%")
    return score


if __name__ == "__main__":
    run()


## Meta-Ensemble Logic

Model-combination and voting logic for comparing or blending predictions.

_Archived from `src/icd10_coding/ensemble/meta_ensemble.py`._


In [ ]:
"""Final ensemble that combines code candidates with chapter agreement."""

import time

import pandas as pd
from scipy.special import expit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

from icd10_coding.ensemble.optimized import (
    add_svm_predictions,
    build_features,
    rebuild_fuzzy_scores,
    train_svm_stack,
)
from icd10_coding.models.classifier import add_classifier_predictions
from icd10_coding.paths import DATA_PROCESSED_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR, ensure_output_dirs


CHAPTER_BONUS = 0.15


def first_char(value) -> str:
    if pd.isna(value) or value == "":
        return ""
    return str(value)[0]


def pick_candidate(row) -> tuple[str, str, float]:
    """Score every model's code and reward candidates that match the predicted chapter."""
    if pd.notna(row.get("exact_code")) and row["exact_code"] != "":
        return row["exact_code"], "exact", 1.0

    candidates = []
    if pd.notna(row.get("fuzzy_code")) and row.get("fuzzy_score", 0) > 0:
        candidates.append([row["fuzzy_score"] / 100.0 * 1.1, row["fuzzy_code"], "fuzzy"])
    if pd.notna(row.get("retrieval_code")):
        candidates.append([row["retrieval_score"] * 1.0, row["retrieval_code"], "retrieval"])
    if pd.notna(row.get("svm_code")) and row["svm_code"] != "":
        candidates.append([row["svm_confidence"] * 1.5, row["svm_code"], "svm_classifier"])
    if pd.notna(row.get("logistic_code")) and row["logistic_code"] != "":
        candidates.append([row["logistic_confidence"] * 1.2, row["logistic_code"], "logistic_classifier"])

    if not candidates:
        return "", "none", 0.0

    # Chapter prediction is not a code by itself; it nudges the full-code candidates.
    predicted_chapter = row.get("predicted_chapter", "")
    for candidate in candidates:
        if first_char(candidate[1]) == predicted_chapter:
            candidate[0] += CHAPTER_BONUS
        else:
            candidate[0] -= CHAPTER_BONUS / 2

    score, code, method = max(candidates, key=lambda item: item[0])
    return code, method, score


def add_retrieval_predictions(df: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    rcv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)
    rwv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
    rcr = rcv.fit_transform(reference["Description_norm"])
    rwr = rwv.fit_transform(reference["Description_norm"])
    vc = rcv.transform(df["Literal_norm"])
    vw = rwv.transform(df["Literal_norm"])

    codes, scores = [], []
    for i in range(0, len(df), 100):
        end = min(i + 100, len(df))
        combined = (cosine_similarity(vc[i:end], rcr) + cosine_similarity(vw[i:end], rwr)) / 2
        best = combined.argmax(axis=1)
        for j in range(end - i):
            codes.append(reference.iloc[best[j]]["Code"])
            scores.append(float(combined[j, best[j]]))
        del combined

    df["retrieval_code"] = codes
    df["retrieval_score"] = scores
    return df


def add_chapter_predictions(df: pd.DataFrame, train: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    train = train.copy()
    reference = reference.copy()
    train["chapter"] = train["Code"].astype(str).str[0]
    reference["chapter"] = reference["Code"].astype(str).str[0]

    texts = pd.concat([train["Literal_norm"], reference["Description_norm"]], ignore_index=True)
    labels = pd.concat([train["chapter"], reference["chapter"]], ignore_index=True)

    vectorizer = build_features()
    x_train = vectorizer.fit_transform(texts)
    classifier = LinearSVC(C=1.0, max_iter=3000, random_state=42)
    classifier.fit(x_train, labels)

    df["predicted_chapter"] = classifier.predict(vectorizer.transform(df["Literal_norm"]))
    return df


def validate() -> float:
    train = pd.read_csv(DATA_PROCESSED_DIR / "train_preprocessed.csv")
    reference = pd.read_csv(DATA_PROCESSED_DIR / "reference_preprocessed.csv")
    train["Literal_norm"] = train["Literal_norm"].fillna("")
    reference["Description_norm"] = reference["Description_norm"].fillna("")

    train_split, val = train_test_split(train, test_size=0.2, random_state=42)
    train_split = train_split.copy()
    val = val.copy()
    val["Literal_norm"] = val["Literal_norm"].fillna("")

    print("Validation: SVM candidates")
    val = add_svm_predictions(val, train_svm_stack(train_split))
    val = val.rename(columns={"clf_code": "svm_code", "clf_confidence": "svm_confidence"})

    print("Validation: logistic candidates")
    val = add_classifier_predictions(val, train_split)
    val = val.rename(columns={"clf_code": "logistic_code", "clf_confidence": "logistic_confidence"})

    print("Validation: fuzzy and retrieval candidates")
    val = rebuild_fuzzy_scores(val, train_split)
    val = add_retrieval_predictions(val, reference)

    print("Validation: chapter candidates")
    val = add_chapter_predictions(val, train_split, reference)

    results = val.apply(pick_candidate, axis=1)
    val["final_code"] = [result[0] for result in results]
    score = accuracy_score(val["Code"], val["final_code"])
    print(f"Meta ensemble validation accuracy: {score * 100:.2f}%")
    return score


def predict_test() -> None:
    optimized = pd.read_csv(PREDICTIONS_DIR / "final_predictions_optimized.csv")
    classifier = pd.read_csv(PREDICTIONS_DIR / "final_predictions.csv")
    chapter = pd.read_csv(PREDICTIONS_DIR / "final_chapter_predictions.csv")

    df = optimized.rename(
        columns={
            "clf_code": "svm_code",
            "clf_confidence": "svm_confidence",
        }
    )
    logistic = classifier[["id", "clf_code", "clf_confidence"]].rename(
        columns={
            "clf_code": "logistic_code",
            "clf_confidence": "logistic_confidence",
        }
    )
    df = df.merge(logistic, on="id", how="left")
    df = df.merge(chapter[["id", "y_category"]], on="id", how="left")
    df = df.rename(columns={"y_category": "predicted_chapter"})

    results = df.apply(pick_candidate, axis=1)
    df["final_code"] = [result[0] for result in results]
    df["final_method"] = [result[1] for result in results]
    df["final_confidence"] = [result[2] for result in results]

    detail_cols = [
        "id", "Literal", "Literal_norm", "predicted_chapter",
        "exact_code", "fuzzy_code", "fuzzy_score",
        "retrieval_code", "retrieval_score",
        "svm_code", "svm_confidence",
        "logistic_code", "logistic_confidence",
        "final_code", "final_method", "final_confidence",
    ]
    df[detail_cols].to_csv(PREDICTIONS_DIR / "final_predictions_meta_ensemble.csv", index=False)

    submission = df[["id", "final_code"]].copy()
    submission.columns = ["id", "Code"]
    submission.to_csv(SUBMISSIONS_DIR / "kaggle_submission_meta_ensemble.csv", index=False)

    print("Meta ensemble method usage:")
    print(df["final_method"].value_counts())
    print("Saved final_predictions_meta_ensemble.csv and kaggle_submission_meta_ensemble.csv")


def run() -> None:
    ensure_output_dirs()
    start = time.time()
    validate()
    predict_test()
    print(f"Done in {time.time() - start:.1f}s")


if __name__ == "__main__":
    run()


## Embedding Experiment Runner

Former command-line wrapper for embedding chapter experiments.

_Archived from `scripts/step6_embedding_chapter.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.models.embedding_chapter import run


if __name__ == "__main__":
    run()


## Meta-Ensemble Runner

Former command-line wrapper for meta-ensemble experiments.

_Archived from `scripts/step7_meta_ensemble.py`._


In [ ]:
from _bootstrap import PROJECT_ROOT  # noqa: F401
from icd10_coding.ensemble.meta_ensemble import run


if __name__ == "__main__":
    run()
